# Session VI: Advanced String Handling and Data Cleaning

Go beyond basic string methods to master advanced text manipulation, pattern matching, and data cleaning techniques essential in real-world applications.

## Exercise 1: Text Normalizer
**Objective:** Practice advanced string methods to clean and standardize messy user input.

You receive raw user input from a registration form. Names have irregular spacing and capitalization, usernames contain unwanted characters, and bios have excessive whitespace. Write functions to normalize each field.

**Tasks:**
1. `normalize_name(name)` → Strip whitespace, collapse multiple spaces into one, and apply title case. Return `None` if the input is not a string or is empty after cleaning.
2. `normalize_username(username)` → Strip, lowercase, and remove any character that is NOT a letter, digit, or underscore. Return `None` if empty after cleaning.
3. `normalize_bio(bio)` → Strip, collapse all whitespace (including tabs and newlines) into single spaces. Truncate to a maximum of 150 characters, adding `"..."` if truncated.
4. `clean_registration(records)` → Take a list of dicts (each with `"name"`, `"username"`, `"bio"`) and return a new list of cleaned dicts. Skip records where either `name` or `username` is invalid.

**Example Input:**
```python
records = [
    {"name": "   aLiCe    sMiTh  ", "username": "  Alice.Smith!  ", "bio": "  Hello!   I love   Python.  "},
    {"name": "", "username": "bob", "bio": "Hi there."},
    {"name": "   Charlie   ", "username": "!!!!", "bio": "A" * 200},
]
```

In [1]:
import re

def normalize_name(name):
    """Normalize a name: strip, collapse spaces, title case."""
    if not isinstance(name, str):
        return None
    # split() with no args strips and collapses all whitespace at once
    name = " ".join(name.split())
    if not name:
        return None
    return name.title()

def normalize_username(username):
    """Normalize a username: strip, lowercase, remove invalid characters."""
    if not isinstance(username, str):
        return None
    username = username.strip().lower()
    username = re.sub(r"[^a-z0-9_]", "", username)
    if not username:
        return None
    return username

def normalize_bio(bio):
    """Normalize a bio: collapse whitespace, truncate to 150 chars."""
    if not isinstance(bio, str):
        return ""
    bio = " ".join(bio.split())
    if len(bio) > 150:
        # 147 chars + "..." so the final bio is still max 150
        bio = bio[:147] + "..."
    return bio

def clean_registration(records):
    """Clean a list of registration records, skipping invalid ones."""
    cleaned = []
    for rec in records:
        name = normalize_name(rec.get("name"))
        username = normalize_username(rec.get("username"))
        if name is None or username is None:
            continue
        cleaned.append({
            "name": name,
            "username": username,
            "bio": normalize_bio(rec.get("bio")),
        })
    return cleaned


# --- TEST ---
print(normalize_name("   aLiCe    sMiTh  "))
# Expected: "Alice Smith"

print(normalize_username("  Alice.Smith!  "))
# Expected: "alicesmith"

print(normalize_bio("  Hello!   I love   Python.  "))
# Expected: "Hello! I love Python."

records = [
    {"name": "   aLiCe    sMiTh  ", "username": "  Alice_99  ", "bio": "  Hello!   I love   Python.  "},
    {"name": "", "username": "bob", "bio": "Hi there."},
    {"name": "   Charlie   ", "username": "!!!!", "bio": "A" * 200},
    {"name": "   Diana   Prince  ", "username": "  wonder_woman  ", "bio": "\t  Saving  \n  the   world.  "},
]
cleaned = clean_registration(records)
for r in cleaned:
    print(r)

Alice Smith
alicesmith
Hello! I love Python.
{'name': 'Alice Smith', 'username': 'alice_99', 'bio': 'Hello! I love Python.'}
{'name': 'Diana Prince', 'username': 'wonder_woman', 'bio': 'Saving the world.'}


---
## Exercise 2: Regex Validator Suite
**Objective:** Write regular expression patterns to validate common data formats.

Build a suite of validator functions using the `re` module. Each function should return `True` if the input matches the expected format, and `False` otherwise.

**Tasks:**
1. `is_valid_email(email)` → Validate an email address: local part (letters, digits, `._%+-`), `@`, domain, `.`, TLD (at least 2 letters).
2. `is_valid_phone(phone)` → Accept any of these formats: `(555) 123-4567`, `555-123-4567`, `555.123.4567`, `5551234567`. Optionally with country code `+1` or `1-` prefix.
3. `is_valid_date(date_str)` → Validate dates in `YYYY-MM-DD` format. Month must be 01-12, day must be 01-31.
4. `is_valid_url(url)` → Validate a basic URL: must start with `http://` or `https://`, followed by a domain and optional path.
5. `validate_record(record)` → Take a dict with `"email"`, `"phone"`, `"date"`, and `"url"` keys. Return a dict mapping each field to `True`/`False`.

In [2]:
import re

EMAIL_PATTERN = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"

def is_valid_email(email):
    """Validate an email address using regex."""
    if not isinstance(email, str):
        return False
    email = email.strip()
    # the basic pattern still accepts double dots like "jack..black@x.com"
    if ".." in email:
        return False
    return bool(re.match(EMAIL_PATTERN, email))

def is_valid_phone(phone):
    """Validate a US phone number in various formats."""
    if not isinstance(phone, str):
        return False
    # only digits and normal separators allowed, otherwise letters would just get stripped
    if not re.match(r"^[\d\s().+-]+$", phone.strip()):
        return False
    digits = re.sub(r"\D", "", phone)
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]
    return len(digits) == 10

def is_valid_date(date_str):
    """Validate a date string in YYYY-MM-DD format."""
    if not isinstance(date_str, str):
        return False
    pattern = r"^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])$"
    return bool(re.match(pattern, date_str.strip()))

def is_valid_url(url):
    """Validate a basic URL (http or https)."""
    if not isinstance(url, str):
        return False
    pattern = r"^https?://[a-zA-Z0-9-]+(\.[a-zA-Z0-9-]+)*\.[a-zA-Z]{2,}(/\S*)?$"
    return bool(re.match(pattern, url.strip()))

def validate_record(record):
    """Validate all fields in a record dict."""
    return {
        "email": is_valid_email(record.get("email")),
        "phone": is_valid_phone(record.get("phone")),
        "date": is_valid_date(record.get("date")),
        "url": is_valid_url(record.get("url")),
    }


# --- TEST ---
print("=== Email ===")
for e in ["alice@example.com", "bob.smith@uni.edu", "missing-at.com", "@no-local.com", "user@domain"]:
    print(f"  {e:>25} → {is_valid_email(e)}")

print("\n=== Phone ===")
for p in ["(555) 123-4567", "555-123-4567", "5551234567", "+1-555-123-4567", "123", "abcdefghij"]:
    print(f"  {p:>20} → {is_valid_phone(p)}")

print("\n=== Date ===")
for d in ["2024-01-15", "2024-13-01", "2024-00-10", "24-01-15", "2024-12-31"]:
    print(f"  {d:>15} → {is_valid_date(d)}")

print("\n=== URL ===")
for u in ["https://example.com", "http://example.com/path", "ftp://wrong.com", "example.com", "https://a.io"]:
    print(f"  {u:>30} → {is_valid_url(u)}")

print("\n=== Record ===")
record = {"email": "alice@example.com", "phone": "(555) 123-4567", "date": "2024-01-15", "url": "https://example.com"}
print(validate_record(record))

=== Email ===
          alice@example.com → True
          bob.smith@uni.edu → True
             missing-at.com → False
              @no-local.com → False
                user@domain → False

=== Phone ===
        (555) 123-4567 → True
          555-123-4567 → True
            5551234567 → True
       +1-555-123-4567 → True
                   123 → False
            abcdefghij → False

=== Date ===
       2024-01-15 → True
       2024-13-01 → False
       2024-00-10 → False
         24-01-15 → False
       2024-12-31 → True

=== URL ===
             https://example.com → True
         http://example.com/path → True
                 ftp://wrong.com → False
                     example.com → False
                    https://a.io → True

=== Record ===
{'email': True, 'phone': True, 'date': True, 'url': True}


---
## Exercise 3: CSV Data Cleaner
**Objective:** Read a messy CSV file, clean every field, and write a clean version.

You receive a CSV file of customer data that is full of problems: inconsistent spacing, mixed capitalization, invalid emails, badly formatted phone numbers, and invalid amounts. Your job is to clean it and produce a pristine output file.

**First, run the cell below to generate the messy CSV file.** Then write the functions to clean it.

**Tasks:**
1. `load_messy_csv(filename)` → Read the CSV with `csv.DictReader`. Return the raw rows as a list of dicts.
2. `clean_name(name)` → Strip, collapse whitespace, apply title case.
3. `clean_email(email)` → Strip, lowercase, validate with regex. Return `None` if invalid.
4. `clean_phone(phone)` → Parse to `(XXX) XXX-XXXX` format. Return `None` if invalid.
5. `clean_amount(amount)` → Remove `$`, `,`, whitespace. Convert to float. Return `0.0` if invalid.
6. `clean_all_rows(rows)` → Apply all cleaners to each row. Return list of cleaned dicts with a new `"is_valid"` boolean field (True if email AND phone are both valid).
7. `save_clean_csv(filename, rows)` → Write the cleaned rows to a new CSV file.
8. `print_report(rows)` → Print a summary: total records, valid records, invalid records, total amount, and a list of invalid entries.

In [3]:
# --- RUN THIS CELL FIRST to generate the messy CSV ---

messy_csv_content = """name,email,phone,amount
   aLiCe   sMiTh  , ALICE@EXAMPLE.COM  , (555) 123-4567 , $1,234.56
  Bob    Johnson, bob.johnson@email.com,555.987.6543,  $890.00
 CHARLIE   WILLIAMS ,charlie-AT-email,  555 111 2222  , $450.75
  diana   PRINCE  , diana.prince@heroes.org , +1-555-333-4444,  2100.50
  Eve    Adams  , eve.adams@  ,  123  ,  $abc  
  FRANK   miller  , frank.miller@work.com , 1-555-777-8888 , $3,456.78
    Grace   HOPPER  ,grace.hopper@navy.mil,5559990000,$12,000.00
  hank   , , ,
  Ivy    Chen  , IVY.CHEN@university.EDU , (555)   444 - 5555 , $  678.90  
  jack    black  , jack..black@movies.com , 555-222-3333 , $1,100.25
"""

with open("messy_customers.csv", "w") as f:
    f.write(messy_csv_content)

print("messy_customers.csv created.")

messy_customers.csv created.


In [4]:
import csv
import re

def load_messy_csv(filename):
    """Read a CSV file and return a list of dicts."""
    rows = []
    with open(filename, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # amounts like $1,234.56 aren't quoted, so the comma splits them into an
            # extra column. DictReader puts the leftovers under the key None -> glue them back
            extra = row.pop(None, None)
            if extra:
                row["amount"] = ",".join([row["amount"]] + extra)
            rows.append(row)
    return rows

def clean_name(name):
    """Strip, collapse whitespace, title case."""
    if not isinstance(name, str):
        return ""
    return " ".join(name.split()).title()

def clean_email(email):
    """Strip, lowercase, validate with regex. Return None if invalid."""
    if not isinstance(email, str):
        return None
    email = email.strip().lower()
    if ".." in email:
        return None
    if re.match(r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$", email):
        return email
    return None

def clean_phone(phone):
    """Parse phone to (XXX) XXX-XXXX format. Return None if invalid."""
    if not isinstance(phone, str):
        return None
    digits = re.sub(r"\D", "", phone)
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]
    if len(digits) != 10:
        return None
    return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"

def clean_amount(amount):
    """Remove $, commas, whitespace. Convert to float. Return 0.0 if invalid."""
    if amount is None:
        return 0.0
    cleaned = re.sub(r"[$,\s]", "", str(amount))
    try:
        return float(cleaned)
    except ValueError:
        return 0.0

def clean_all_rows(rows):
    """Apply all cleaners to each row. Add 'is_valid' field."""
    cleaned = []
    for row in rows:
        email = clean_email(row.get("email"))
        phone = clean_phone(row.get("phone"))
        cleaned.append({
            "name": clean_name(row.get("name")),
            "email": email,
            "phone": phone,
            "amount": clean_amount(row.get("amount")),
            "is_valid": email is not None and phone is not None,
        })
    return cleaned

def save_clean_csv(filename, rows):
    """Write cleaned rows to a new CSV file."""
    if not rows:
        return
    with open(filename, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        for row in rows:
            # write blank instead of the word "None"
            writer.writerow({k: ("" if v is None else v) for k, v in row.items()})

def print_report(rows):
    """Print a cleaning summary report."""
    valid = [r for r in rows if r["is_valid"]]
    invalid = [r for r in rows if not r["is_valid"]]
    total_amount = sum(r["amount"] for r in rows)

    print("=" * 40)
    print("CUSTOMER DATA CLEANING REPORT")
    print("=" * 40)
    print(f"Total records:   {len(rows)}")
    print(f"Valid records:   {len(valid)}")
    print(f"Invalid records: {len(invalid)}")
    print(f"Total amount:    ${total_amount:,.2f}")
    print("\nInvalid entries:")
    for r in invalid:
        problems = []
        if r["email"] is None:
            problems.append("bad email")
        if r["phone"] is None:
            problems.append("bad phone")
        print(f"  - {r['name']:<18} {', '.join(problems)}")


# --- PROCESS ---
raw_rows = load_messy_csv("messy_customers.csv")
cleaned = clean_all_rows(raw_rows)
save_clean_csv("clean_customers.csv", cleaned)
print_report(cleaned)

CUSTOMER DATA CLEANING REPORT
Total records:   10
Valid records:   6
Invalid records: 4
Total amount:    $21,911.74

Invalid entries:
  - Charlie Williams   bad email
  - Eve Adams          bad email, bad phone
  - Hank               bad email, bad phone
  - Jack Black         bad email


---
## Exercise 4: Phone Number Parser
**Objective:** Build a robust phone number cleaning function that handles many different input formats.

You receive a list of contact records with phone numbers in wildly different formats. Write a parser that normalizes them all to a single consistent format, and a batch processor to clean an entire contact list.

**Tasks:**
1. `parse_phone(raw_phone)` → Accept any reasonable phone format and return `(XXX) XXX-XXXX`. Handle optional country code `+1` or `1-`. Return `None` for invalid numbers (not 10 digits after cleaning).
2. `extract_phone_parts(formatted_phone)` → Take a formatted phone `(XXX) XXX-XXXX` and return a dict with `"area_code"`, `"prefix"`, and `"line"` keys.
3. `batch_clean_phones(contacts)` → Take a list of dicts with `"name"` and `"phone"` keys. Return a new list with cleaned phones and a `"valid"` boolean.
4. `group_by_area_code(contacts)` → Take the cleaned contacts and return a dict mapping each area code to a list of names.
5. `print_phone_report(contacts)` → Print a formatted report showing valid/invalid counts and groupings by area code.

In [5]:
import re

def parse_phone(raw_phone):
    """Parse any phone format to (XXX) XXX-XXXX. Return None if invalid."""
    if not isinstance(raw_phone, str):
        return None
    digits = re.sub(r"\D", "", raw_phone)
    # +1 / 1- country code -> 11 digits starting with 1
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]
    if len(digits) != 10:
        return None
    return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"

def extract_phone_parts(formatted_phone):
    """Extract area_code, prefix, and line from (XXX) XXX-XXXX."""
    if not formatted_phone:
        return None
    match = re.match(r"^\((\d{3})\) (\d{3})-(\d{4})$", formatted_phone)
    if not match:
        return None
    return {
        "area_code": match.group(1),
        "prefix": match.group(2),
        "line": match.group(3),
    }

def batch_clean_phones(contacts):
    """Clean all phone numbers in a contact list."""
    result = []
    for c in contacts:
        phone = parse_phone(c.get("phone"))
        result.append({
            "name": c.get("name"),
            "phone": phone,
            "original": c.get("phone"),
            "valid": phone is not None,
        })
    return result

def group_by_area_code(contacts):
    """Group valid contacts by area code."""
    groups = {}
    for c in contacts:
        if not c["valid"]:
            continue
        area = extract_phone_parts(c["phone"])["area_code"]
        if area not in groups:
            groups[area] = []
        groups[area].append(c["name"])
    return groups

def print_phone_report(contacts):
    """Print a summary of cleaned phone data."""
    valid = [c for c in contacts if c["valid"]]
    invalid = [c for c in contacts if not c["valid"]]

    print("=" * 45)
    print("PHONE CLEANING REPORT")
    print("=" * 45)
    for c in contacts:
        shown = c["phone"] if c["valid"] else "INVALID"
        print(f"  {c['name']:<10} {c['original'].strip():<22} -> {shown}")
    print(f"\nValid: {len(valid)}   Invalid: {len(invalid)}")
    if invalid:
        print("Invalid contacts:", ", ".join(c["name"] for c in invalid))

    print("\nBy area code:")
    for area, names in sorted(group_by_area_code(contacts).items()):
        print(f"  {area}: {', '.join(names)} ({len(names)})")


# --- TEST ---
contacts = [
    {"name": "Alice", "phone": "(555) 123-4567"},
    {"name": "Bob", "phone": "555.987.6543"},
    {"name": "Charlie", "phone": "+1-555-111-2222"},
    {"name": "Diana", "phone": "5559990000"},
    {"name": "Eve", "phone": "123"},
    {"name": "Frank", "phone": "1-800-555-0199"},
    {"name": "Grace", "phone": "(800) 555-0100"},
    {"name": "Hank", "phone": "not a phone number"},
    {"name": "Ivy", "phone": "  (555)   444 - 5555  "},
    {"name": "Jack", "phone": "+1 800 555 0150"},
]

cleaned = batch_clean_phones(contacts)
print_phone_report(cleaned)
groups = group_by_area_code(cleaned)
print("\nGrouped by area code:", groups)

PHONE CLEANING REPORT
  Alice      (555) 123-4567         -> (555) 123-4567
  Bob        555.987.6543           -> (555) 987-6543
  Charlie    +1-555-111-2222        -> (555) 111-2222
  Diana      5559990000             -> (555) 999-0000
  Eve        123                    -> INVALID
  Frank      1-800-555-0199         -> (800) 555-0199
  Grace      (800) 555-0100         -> (800) 555-0100
  Hank       not a phone number     -> INVALID
  Ivy        (555)   444 - 5555     -> (555) 444-5555
  Jack       +1 800 555 0150        -> (800) 555-0150

Valid: 8   Invalid: 2
Invalid contacts: Eve, Hank

By area code:
  555: Alice, Bob, Charlie, Diana, Ivy (5)
  800: Frank, Grace, Jack (3)

Grouped by area code: {'555': ['Alice', 'Bob', 'Charlie', 'Diana', 'Ivy'], '800': ['Frank', 'Grace', 'Jack']}


---
## Exercise 5: Encoding Fixer
**Objective:** Handle text files with encoding problems, special characters, and non-printable content.

You receive text files from various sources (old systems, web scraping, international partners). They contain encoding artifacts, non-printable characters, and inconsistent Unicode. Write functions to clean them.

**First, run the cell below to generate sample files.** Then write the cleaning functions.

**Tasks:**
1. `read_safe(filename, encodings)` → Try reading a file with each encoding in the list. Return the content from the first encoding that works. Return `None` if all fail.
2. `remove_non_printable(text)` → Remove all non-printable characters (keep only printable ASCII + newlines and tabs).
3. `fix_common_artifacts(text)` → Replace common encoding artifacts: `Ã©` → `é`, `Ã±` → `ñ`, `â€™` → `'`, `â€œ` → `"`, `â€\x9d` → `"`, `Ã¼` → `ü`.
4. `normalize_unicode(text)` → Use `unicodedata.normalize("NFC", ...)` to normalize Unicode characters.
5. `clean_file(input_file, output_file, encodings)` → Read, clean (remove non-printable + fix artifacts + normalize), and write the cleaned version to a new file.

In [6]:
# --- RUN THIS CELL FIRST to generate sample files ---

# File 1: Text with non-printable characters
with open("messy_text_1.txt", "w", encoding="utf-8") as f:
    f.write("Hello\x00 World\x07!\n")
    f.write("This has\x1b control\x0b characters.\n")
    f.write("Normal line here.\n")
    f.write("Another\x00 messy\x07 line.\n")

# File 2: Text with encoding artifacts
with open("messy_text_2.txt", "w", encoding="utf-8") as f:
    f.write("The cafÃ© is on the corner.\n")
    f.write("She said â€œhelloâ€\x9d to everyone.\n")
    f.write("The niño played in the señora's garden.\n")
    f.write("MÃ¼nchen is beautiful in spring.\n")
    f.write("Itâ€™s a wonderful day.\n")

# File 3: Latin-1 encoded file
# NOTE: the original cell used encoding="latin-1", but latin-1 has no € sign so it
# crashes with UnicodeEncodeError. cp1252 (Windows Latin-1) is the same plus €.
with open("messy_text_3.txt", "w", encoding="cp1252") as f:
    f.write("Résumé of José García\n")
    f.write("Worked at Ströer & Müller GmbH\n")
    f.write("Salary: €85,000\n")

print("Sample files created: messy_text_1.txt, messy_text_2.txt, messy_text_3.txt")

Sample files created: messy_text_1.txt, messy_text_2.txt, messy_text_3.txt


In [7]:
import re
import unicodedata

def read_safe(filename, encodings=None):
    """Try reading a file with multiple encodings. Return content or None."""
    if encodings is None:
        # cp1252 before latin-1: latin-1 never fails (every byte maps to something),
        # so anything after it in the list would never be tried
        encodings = ["utf-8", "cp1252", "latin-1", "ascii"]
    for enc in encodings:
        try:
            with open(filename, "r", encoding=enc) as f:
                return f.read()
        except (UnicodeDecodeError, LookupError):
            continue
    return None

def remove_non_printable(text):
    """Remove non-printable characters, keep printable ASCII + newline + tab."""
    # remove control characters but keep accented letters like é, ñ, ü
    return re.sub(r"[\x00-\x08\x0b-\x1f\x7f-\x9f]", "", text)

def fix_common_artifacts(text):
    """Replace common encoding artifacts with correct characters."""
    # longer artifacts first so they don't get partially replaced
    artifacts = {
        "â€œ": '"',
        "â€\x9d": '"',
        "â€™": "'",
        "Ã©": "é",
        "Ã±": "ñ",
        "Ã¼": "ü",
    }
    for bad, good in artifacts.items():
        text = text.replace(bad, good)
    return text

def normalize_unicode(text):
    """Normalize Unicode to NFC form."""
    return unicodedata.normalize("NFC", text)

def clean_file(input_file, output_file, encodings=None):
    """Read, clean, and save a text file."""
    content = read_safe(input_file, encodings)
    if content is None:
        print(f"Could not read {input_file}")
        return
    # fix artifacts BEFORE removing non-printables, because the closing quote
    # artifact contains \x9d, which remove_non_printable would delete first
    content = fix_common_artifacts(content)
    content = remove_non_printable(content)
    content = normalize_unicode(content)
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"{input_file} -> {output_file}")


# --- TEST ---
# Test read_safe
content = read_safe("messy_text_3.txt")
print("Read with fallback encoding:")
print(content)

# Test cleaning pipeline
clean_file("messy_text_1.txt", "clean_text_1.txt")
clean_file("messy_text_2.txt", "clean_text_2.txt")
clean_file("messy_text_3.txt", "clean_text_3.txt")

# Verify
for fname in ["clean_text_1.txt", "clean_text_2.txt", "clean_text_3.txt"]:
    print(f"\n=== {fname} ===")
    with open(fname, "r", encoding="utf-8") as f:
        print(f.read())

Read with fallback encoding:
Résumé of José García
Worked at Ströer & Müller GmbH
Salary: €85,000

messy_text_1.txt -> clean_text_1.txt
messy_text_2.txt -> clean_text_2.txt
messy_text_3.txt -> clean_text_3.txt

=== clean_text_1.txt ===
Hello World!
This has control characters.
Normal line here.
Another messy line.


=== clean_text_2.txt ===
The café is on the corner.
She said "hello" to everyone.
The niño played in the señora's garden.
München is beautiful in spring.
It's a wonderful day.


=== clean_text_3.txt ===
Résumé of José García
Worked at Ströer & Müller GmbH
Salary: €85,000



---
## Exercise 6: Data Cleaning Module
**Objective:** Build a complete, reusable data cleaning module and use it to process a messy dataset.

This exercise combines everything from this session. You will:
1. Define a set of **reusable cleaning functions** (like a `cleaners` module).
2. Define a **configurable cleaning pipeline** that applies different cleaners to different columns.
3. Process a messy employee dataset and produce a clean CSV + summary report.

**First, run the cell below to generate the messy dataset.** Then build your cleaning pipeline.

**Tasks:**
1. **Cleaning functions** (reusable, no `print()` inside):
   - `clean_text(text)` → Strip, collapse whitespace, lowercase.
   - `clean_name(name)` → Strip, collapse whitespace, title case. Return `"Unknown"` if empty/invalid.
   - `clean_email(email)` → Strip, lowercase, validate with regex. Return `None` if invalid.
   - `clean_phone(phone)` → Parse to `(XXX) XXX-XXXX`. Return `None` if invalid.
   - `clean_salary(salary)` → Remove `$`, `,`, whitespace. Convert to float. Return `0.0` if invalid.
   - `clean_department(dept, valid_depts)` → Normalize and match against a list of valid departments. Return `"Other"` if no match.

2. **Pipeline function:**
   - `clean_employee_record(record, valid_depts)` → Apply the correct cleaner to each field. Add an `"is_valid"` field (True if name ≠ "Unknown" AND email is not None AND phone is not None).

3. **Batch processing:**
   - `process_employee_file(input_file, output_file, valid_depts)` → Read messy CSV, clean all records, write clean CSV, print summary report.

4. **Summary report** should include:
   - Total records processed
   - Valid vs. invalid counts (with percentage)
   - Breakdown by department
   - Total salary sum
   - List of invalid records with the reason why

In [8]:
# --- RUN THIS CELL FIRST to generate the messy employee dataset ---

messy_employee_data = """name,email,phone,department,salary
   ALICE   SMITH  , ALICE.SMITH@COMPANY.COM  , (555) 123-4567 ,  engineering  , $95,000.00
  bob    JOHNSON, bob.johnson@company.com,555.987.6543,  MARKETING  ,  $82,500.00
 CHARLIE   WILLIAMS ,charlie-AT-email,  555 111 2222  ,  Engineering , $78,000.00
  diana   PRINCE  , diana.prince@company.com , +1-555-333-4444, Marketing ,$105,000.00
  , , ,  ,
  Eve    Adams  , eve.adams@  ,  123  ,  SALES  ,  $abc  
  FRANK   miller  , frank.miller@company.com , 1-555-777-8888 , sales , $91,200.00
    Grace   HOPPER  ,grace.hopper@company.com,5559990000,  ENGINEERING  ,$120,000.00
  hank   PATEL  , hank.patel@company.com, (555) 222-3333 , Mktg , $67,800.00
  Ivy    CHEN  , IVY.CHEN@COMPANY.COM , (555)   444 - 5555 ,  eng  , $  88,500.00  
  jack    BLACK  , jack.black@company.com , 555-666-7777 , Human Resources , $72,000.00
  Kate   , kate@company.com , 555-888-9999 ,  HR  , $68,000.00
"""

with open("messy_employees.csv", "w") as f:
    f.write(messy_employee_data)

print("messy_employees.csv created.")

messy_employees.csv created.


In [9]:
import csv
import re

# --- VALID DEPARTMENTS (with common aliases) ---
DEPARTMENT_MAP = {
    "engineering": "Engineering",
    "eng": "Engineering",
    "marketing": "Marketing",
    "mktg": "Marketing",
    "sales": "Sales",
    "human resources": "Human Resources",
    "hr": "Human Resources",
}

# --- CLEANING FUNCTIONS ---

def clean_text(text):
    """Strip, collapse whitespace, lowercase."""
    if not isinstance(text, str):
        return ""
    return " ".join(text.split()).lower()

def clean_name(name):
    """Strip, collapse whitespace, title case. Return 'Unknown' if invalid."""
    name = clean_text(name)
    if not name:
        return "Unknown"
    return name.title()

def clean_email(email):
    """Strip, lowercase, validate. Return None if invalid."""
    email = clean_text(email).replace(" ", "")
    if ".." in email:
        return None
    if re.match(r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$", email):
        return email
    return None

def clean_phone(phone):
    """Parse to (XXX) XXX-XXXX. Return None if invalid."""
    if not isinstance(phone, str):
        return None
    digits = re.sub(r"\D", "", phone)
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]
    if len(digits) != 10:
        return None
    return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"

def clean_salary(salary):
    """Remove $, commas, spaces. Convert to float. Return 0.0 if invalid."""
    if salary is None:
        return 0.0
    try:
        return float(re.sub(r"[$,\s]", "", str(salary)))
    except ValueError:
        return 0.0

def clean_department(dept, dept_map):
    """Normalize and match against valid departments. Return 'Other' if no match."""
    return dept_map.get(clean_text(dept), "Other")


# --- PIPELINE ---

def clean_employee_record(record, dept_map):
    """Clean all fields in an employee record."""
    cleaned = {
        "name": clean_name(record.get("name")),
        "email": clean_email(record.get("email")),
        "phone": clean_phone(record.get("phone")),
        "department": clean_department(record.get("department"), dept_map),
        "salary": clean_salary(record.get("salary")),
    }
    cleaned["is_valid"] = (
        cleaned["name"] != "Unknown"
        and cleaned["email"] is not None
        and cleaned["phone"] is not None
    )
    return cleaned


def invalid_reasons(rec):
    """Helper for the report: list what is wrong with a record."""
    reasons = []
    if rec["name"] == "Unknown":
        reasons.append("missing name")
    if rec["email"] is None:
        reasons.append("invalid email")
    if rec["phone"] is None:
        reasons.append("invalid phone")
    return reasons


# --- BATCH PROCESSING ---

def process_employee_file(input_file, output_file, dept_map):
    """Read messy CSV, clean all records, write clean CSV, print report."""
    # read
    with open(input_file, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        raw_rows = []
        for row in reader:
            # same issue as Ex 3: "$95,000.00" isn't quoted so it spills into an extra column
            extra = row.pop(None, None)
            if extra:
                row["salary"] = ",".join([row["salary"]] + extra)
            raw_rows.append(row)

    # clean
    cleaned = [clean_employee_record(r, dept_map) for r in raw_rows]

    # write
    fields = ["name", "email", "phone", "department", "salary", "is_valid"]
    with open(output_file, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for rec in cleaned:
            writer.writerow({k: ("" if rec[k] is None else rec[k]) for k in fields})

    # report
    total = len(cleaned)
    valid = [r for r in cleaned if r["is_valid"]]
    invalid = [r for r in cleaned if not r["is_valid"]]

    dept_counts = {}
    dept_salary = {}
    for r in cleaned:
        d = r["department"]
        dept_counts[d] = dept_counts.get(d, 0) + 1
        dept_salary[d] = dept_salary.get(d, 0) + r["salary"]

    print("=" * 50)
    print("EMPLOYEE DATA CLEANING REPORT")
    print("=" * 50)
    print(f"Total records processed: {total}")
    print(f"Valid:   {len(valid):>2} ({len(valid) / total * 100:.1f}%)")
    print(f"Invalid: {len(invalid):>2} ({len(invalid) / total * 100:.1f}%)")

    print("\nBy department:")
    for d in sorted(dept_counts):
        label = "employee" if dept_counts[d] == 1 else "employees"
        print(f"  {d:<17} {dept_counts[d]:>2} {label:<9}   ${dept_salary[d]:>11,.2f}")

    print(f"\nTotal salary: ${sum(r['salary'] for r in cleaned):,.2f}")

    print("\nInvalid records:")
    for r in invalid:
        print(f"  - {r['name']:<17} -> {', '.join(invalid_reasons(r))}")

    print(f"\nClean file saved to {output_file}")
    return cleaned


# --- RUN ---
employees = process_employee_file("messy_employees.csv", "clean_employees.csv", DEPARTMENT_MAP)

EMPLOYEE DATA CLEANING REPORT
Total records processed: 12
Valid:    9 (75.0%)
Invalid:  3 (25.0%)

By department:
  Engineering        4 employees   $ 381,500.00
  Human Resources    2 employees   $ 140,000.00
  Marketing          3 employees   $ 255,300.00
  Other              1 employee    $       0.00
  Sales              2 employees   $  91,200.00

Total salary: $868,000.00

Invalid records:
  - Charlie Williams  -> invalid email
  - Unknown           -> missing name, invalid email, invalid phone
  - Eve Adams         -> invalid email, invalid phone

Clean file saved to clean_employees.csv
